# N126 · 欧几里得算法与整除结构

**目标：** 从余数不改变公因子推导gcd及应用。

**先修：** N002, N013, N063。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** gcd/lcm；扩展欧几里得；丢番图方程；符号与零；约数。

**配套讲解来源：** [同名逐章意见](../../comment/126_gcd_euclid_number_theory.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这章从"最大公约数"这个古老问题出发，一路讲到三个现代应用：既能算数，又能算字符串。

1. 求两个整数的最大公约数（gcd）。例子：gcd(252, 105) = 21，因为 252 = 2²×3²×7、105 = 3×5×7，公共质因子的最小组合是 3×7 = 21。暴力要试除到较小的那个数；欧几里得（辗转相除）三步就收工。
2. 找出 x、y 使 a·x + b·y = gcd(a,b)（Bézout 系数），这也是解丢番图方程 ax+by=c 的钥匙。例子：252×(−2) + 105×5 = −504 + 525 = 21，所以 x=−2、y=5 是一组解。
3. 求两个字符串的"公因子"：最长的字符串 t，使得 a 和 b 都由 t 重复拼接而成。例子：a = 'ABABAB'、b = 'ABAB'，答案是 'AB'；而 a = 'LEET'、b = 'CODE' 没有公因子，答案是空串。注意这不是"字符集合求交集"，而是"共同的重复基本块"。

一句话主线（也是 notebook 的推导核心）：做一次带余除法 a = q·b + r 之后，"a 和 b 的公约数集合"与"b 和 r 的公约数集合"完全相同，所以余数换进来、答案不变，反复做到余数为 0，剩下的就是 gcd。

## 2. 基础知识：先读懂这些词

- **整除与约数**：d 整除 a（写作 d | a）意思是 a 除以 d 没有余数。a 的约数就是所有能整除 a 的正整数。公约数是同时整除 a 和 b 的数。
- **gcd（最大公约数）**：公约数中最大的那个。它有一个很强的性质：任何公约数都整除 gcd——这也是"最大"的本质。本章约定 gcd(0,0) = 0（和 Python 的 math.gcd 一致）。
- **lcm（最小公倍数）**：同时是 a、b 倍数的最小正数。它与 gcd 通过公式 |a·b| = gcd(a,b) · lcm(a,b) 相连，所以算完 gcd 顺手就能算 lcm。本章约定含 0 的 lcm 为 0。
- **欧几里得算法（辗转相除）**：反复用 (a, b) → (b, a mod b) 替换，直到第二个数变 0，此时第一个数就是 gcd。它的正确性靠"余数不改变公约数集合"这条性质。
- **带余除法与不变量**：a = q·b + r（q 是商、r 是余数）。反过来 r = a − q·b，这说明 r 是 a 和 b 的整数线性组合——"可逆的线性组合"正是公约数集合不变的原因，也是扩展算法的入口。
- **Bézout 恒等式与系数**：存在整数 x、y 使 a·x + b·y = gcd(a,b)，这组 (x,y) 叫 Bézout 系数。它不唯一（比如同加减 b/g、a/g 可以造出无穷多组）。
- **扩展欧几里得**：在辗转相除的同时，为每个余数维护"它是 a、b 的哪种线性组合"，做完时顺手读出 Bézout 系数的算法。核心不变量：每一轮都有 r = |a|·x + |b|·y。
- **丢番图方程**：ax + by = c 的整数解问题。它有解当且仅当 gcd(a,b) 整除 c；有解时先用扩展欧几里得拿一组基解再放大。
- **符号与零的约定**：gcd 按绝对值定义（gcd(−4,6)=2）；gcd(x,0)=|x|；lcm 遇 0 记 0。这些边界不约定清楚，代码就会在负数和零上翻车。
- **字符串的"公因子"**：字符串 t 是 a 的"因子"，指 a 由 t 重复若干次拼成（如 'ABAB' = 'AB'×2）。两串的公共重复块最长是 gcd(len(a), len(b))——前提是两串真的同源，这需要一个额外判据（见下）。
- **拼接可交换判据**：a+b == b+a 当且仅当 a、b 由同一个基本块重复组成。这是字符串 gcd 的守门条件，缺了它单看长度 gcd 会出错。

## 3. 思路推导：从小例子开始

- Step 1：为什么余数不改变公约数集合。设 d 整除 a 和 b，又 r = a − q·b，那么 d 也整除 r；反过来设 d 整除 b 和 r，由 a = q·b + r 知 d 也整除 a。两个方向的公约数完全一样，所以把 a 换成 b、b 换成 r，问题原地不动、规模却在缩小。
- Step 2：手算辗转相除。取 a=252、b=105（下面这张表就是 notebook “运行示例”部分 打印的过程）：

| a | b | 商 a//b | 余数 a%b |
|---|---|---|---|
| 252 | 105 | 2 | 42 |
| 105 | 42 | 2 | 21 |
| 42 | 21 | 2 | 0 |

  第三行余数为 0，所以 gcd = 21。注意每行都是"a = 商×b + 余数"：252 = 2×105 + 42；105 = 2×42 + 21；42 = 2×21 + 0。
- Step 3：扩展算法在这张表旁边加两列系数。不变量是"当前余数 r = 252·x + 105·y"：
  - 初始：252 = 252×1 + 105×0（x=1,y=0）；105 = 252×0 + 105×1（x=0,y=1）。
  - 42 = 252 − 2×105 = 252×1 + 105×(−2)。
  - 21 = 105 − 2×42 = 252×0 + 105×1 − 2×(252×1 + 105×(−2)) = 252×(−2) + 105×5。
  - 停在 21：读出 Bézout 系数 x=−2、y=5，正是 notebook 打印的 `252×-2+105×5=21`。
- Step 4：lcm 用公式。gcd(252,105)=21，lcm = 252×105/21 = 252×5 = 1260。代码先除后乘（a//g 再乘 b），避免中间乘出一个巨大的数。
- Step 5：字符串 gcd 的两步走。先判 a+b == b+a：'ABABAB'+'ABAB' 与 'ABAB'+'ABABAB' 都是 'ABABABABAB'，通过；'LEET'+'CODE' = 'LEETCODE' 而 'CODE'+'LEET' = 'CODELEET'，不等，直接返回空串。通过者取长度 gcd：gcd(6,4) = 2，答案就是 a 的前 2 个字符 'AB'。为什么要先判？反例：a='ABACADA'、b='A'，长度 gcd 是 1，直接切前缀会返回 'A'，但 'ABACADA' 根本不是 'A' 重复出来的——拼接可交换这一关就把假货挡在门外。

## 4. 核心代码：gcd_euclid

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def gcd_euclid(a, b):
    a, b = (abs(a), abs(b))
    while b:
        a, b = (b, a % b)
    return a
```

### 逐段读懂代码

### 4.1 gcd_euclid：辗转相除

```python
def gcd_euclid(a, b):
    a, b = (abs(a), abs(b))
    while b:
        a, b = (b, a % b)
    return a
```

- `a,b=abs(a),abs(b)` 先取绝对值：gcd 按定义只看大小不看正负，这一行让负数输入直接归一。
- `while b: a,b=b,a%b` 是算法全部：b 非 0 就做一轮"新 a = 旧 b，新 b = 旧 a 除旧 b 的余数"。b 变 0 时，a 就是 gcd。b=0 一开始就成立的话（含 gcd(0,0)），循环直接跳过，返回 a——gcd(x,0)=|x| 与 gcd(0,0)=0 两个约定自动满足。

### 4.2 extended_gcd：边做除法边记账

```python
def extended_gcd(a, b):
    old_r, r = (abs(a), abs(b))
    old_x, x = (1, 0)
    old_y, y = (0, 1)
    while r:
        q = old_r // r
        old_r, r = (r, old_r - q * r)
        old_x, x = (x, old_x - q * x)
        old_y, y = (y, old_y - q * y)
    return (old_r, old_x * (1 if a >= 0 else -1), old_y * (1 if b >= 0 else -1))
```

- 六个变量其实是三对"(值, 系数)"：old_r/r 是上一轮、本轮的余数，(old_x,old_y)、(x,y) 分别是它们的 Bézout 系数。初始把 |a| 和 |b| 本身写成线性组合：|a| = |a|×1 + |b|×0、|b| = |a|×0 + |b|×1。
- 每轮先算商 q=old_r//r；然后三组同步更新，全部套用同一个模式"新值 = 旧值 − q×更旧值"。余数那一组是 old_r − q·r = 余数（带余除法），系数两组做完全相同的线性组合，于是不变量 r = |a|·x + |b|·y 一路保持——这就是第三节 Step 3 手算过程的一般化。
- 循环结束（r=0）时 old_r 是 gcd，(old_x, old_y) 是针对 |a|、|b| 的系数。返回前那两串 `(1 if a>=0 else -1)` 是符号修正：如果原输入 a 是负数，等式 |a|·old_x = a·(−old_x)，所以 x 的符号要翻过来；b 同理。a、b 同号非负时修正系数是 1，什么都不变。

### 4.3 lcm：先除后乘

```python
def lcm(a, b):
    return abs(a // gcd_euclid(a, b) * b) if a and b else 0
```

- `if a and b else 0` 是零约定：任一输入为 0，lcm 记 0（这也和 math.lcm 一致），同时躲开了除以 gcd(0,0)=0 的危险。
- 主表达式先 `a//gcd_euclid(a,b)` 再乘 b：先除后乘让中间结果保持和最终值同一量级；如果先 a*b 再除，两个大数相乘可能造出天文数字。最外层 abs 处理负号。

### 4.4 gcd_of_strings：字符串公因子

```python
def gcd_of_strings(a, b):
    if not a or not b or a + b != b + a:
        return ''
    return a[:gcd_euclid(len(a), len(b))]
```

- 第一行是三重门：任一串为空，返回空串；两串拼接不可交换（a+b != b+a），说明它们没有共同基本块，也返回空串。这一行是整个函数的灵魂判据。
- 通过后，两串必是同一基本块的重复，该块的最大长度就是两串长度的 gcd，所以直接切 `a[:gcd_euclid(len(a),len(b))]`。数学依据（“正确性与复杂度”部分 的论证）：若 a·b = b·a 且都非空，则存在原始串 t 使 a=t^m、b=t^n，两长度的 gcd 恰是 t 的长度。

## 5. 分段实现：按顺序运行

**本章接口：** `gcd_euclid(a, b)`、`extended_gcd(a, b)`、`lcm(a, b)`、`gcd_of_strings(a, b)`

### gcd_euclid

In [1]:
def gcd_euclid(a, b):
    a, b = (abs(a), abs(b))
    while b:
        a, b = (b, a % b)
    return a

### extended_gcd

In [2]:
def extended_gcd(a, b):
    old_r, r = (abs(a), abs(b))
    old_x, x = (1, 0)
    old_y, y = (0, 1)
    while r:
        q = old_r // r
        old_r, r = (r, old_r - q * r)
        old_x, x = (x, old_x - q * x)
        old_y, y = (y, old_y - q * y)
    return (old_r, old_x * (1 if a >= 0 else -1), old_y * (1 if b >= 0 else -1))

### lcm

In [3]:
def lcm(a, b):
    return abs(a // gcd_euclid(a, b) * b) if a and b else 0

### gcd_of_strings

In [4]:
def gcd_of_strings(a, b):
    if not a or not b or a + b != b + a:
        return ''
    return a[:gcd_euclid(len(a), len(b))]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a,b=252,105; rows=[]
while b: rows.append((a,b,a//b,a%b)); a,b=b,a%b
show_table(['a','b','商','余数'],rows)
g,x,y=extended_gcd(252,105); print(f'252×{x}+105×{y}={g}')

a,b,商,余数
252,105,2,42
105,42,2,21
42,21,2,0


252×-2+105×5=21


## 7. 正确性、适用条件与复杂度

余数变换是可逆的整数线性组合，因此公约数不变。扩展算法每轮保留r=a·x+b·y的不变量。非空字符串若ab=ba，则都由同一原始串重复组成，公共重复块最大长度为两长度的最大公约数。

**代价：** 整数GCD执行O(log min(|a|,|b|))级别除法次数，实际大整数除法位复杂度另计；字符串拼接比较O(|a|+|b|)时间空间。约定gcd(0,0)=0，含0的lcm=0；空字符串扩展返回空串。

### 展开理解

正确性三段论：

- 欧几里得为什么对：Step 1 已证"公约数集合在 (a,b)→(b,a%b) 下完全不变"，所以整条链上每一对数的公约数集合都等于最初那对；最后一对是 (g, 0)，而任何数和 0 的公约数就是它自身的约数，所以 g 的约数 = 原来的公约数，g 本身就是最大的那个。
- 扩展算法为什么对：每轮更新对余数和系数做同一个线性组合，所以"r = |a|·x + |b|·y"在每一轮都成立；结束时 r=0、old_r=g，就有 g = |a|·old_x + |b|·old_y，再按输入符号翻转系数，等式对原符号的 a、b 依然成立。注意解不唯一——这只是一组特解，测试也只验证"这组解满足等式"，不验证唯一。
- 字符串 gcd 为什么对：拼接可交换是两串同源的判据（充分必要）。过了这一关，a、b 都是某基本块 t 的重复，块长必同时整除两个长度，所以块长 ≤ gcd(len)；而长度为 gcd 的前缀本身就是合法基本块（可以由周期性论证），于是"切前缀取 gcd 长"恰好拿到最大的。

复杂度（结合具体规模感受）：

- 欧几里得：每次带余除法至少把较大的数砍掉一半以上（最坏情况由相邻斐波那契数给出，每两轮余数至少缩到原来的约 0.618²），所以除法次数是 O(log min(|a|,|b|)) 级别。经典的经验法则（Lamé 定理）说：除法次数不超过较小数十进制位数的 5 倍。举例感受：gcd(75025, 46368)（这是相邻斐波那契数 F25、F24，公认的最坏输入）也就二十来次除法；a、b 取到 10¹⁸（64 位上限）时最坏约八九十次，眨眼完成。注意 Python 大整数的单次除法本身要按位数计成本，算天文数字时另算。
- 扩展欧几里得：与普通版同轮数，每轮多做几次乘加，仍是 O(log min) 轮。
- 字符串 gcd：拼接两次、比较一次是 O(|a|+|b|)，长度 gcd 便宜到忽略不计，总体 O(|a|+|b|) 时间和空间。

顺带把知识点清单里的"丢番图方程"落到数字上：ax + by = c 有整数解，当且仅当 gcd(a,b) 整除 c。用法是先拿扩展欧几里得的一组基解再放大。手算例：252x + 105y = 42 有解吗？g=21 整除 42，有解；把 Bézout 解 (−2,5) 两边同乘 2 得 (−4,10)，验算 252×(−4) + 105×10 = −1008 + 1050 = 42，成立。反过来 252x + 105y = 41 无解，因为 21 不整除 41——左边任何组合都是 21 的倍数。

零与符号的约定再念一遍（“正确性与复杂度”部分 明文）：gcd(0,0)=0；gcd(x,0)=|x|；含 0 的 lcm=0；gcd/lcm 均按绝对值；扩展算法返回的等式对任意符号输入都成立；空字符串输入返回空串。

## 8. 单元测试：每个用例在检查什么

```python
assert gcd_euclid(54, 24) == 6 and gcd_euclid(0, 0) == 0
```
前半条是教科书正常值：54=2×3²×3? 实际 54=2×3³、24=2³×3，公共部分 2×3=6。后半条专测零约定：math.gcd 也定义 gcd(0,0)=0，代码里循环直接不进、返回 abs(0)=0，契约对齐。

```python
assert gcd_of_strings('ABCABC', 'ABC') == 'ABC'

assert gcd_of_strings('ABABAB', 'ABAB') == 'AB' and gcd_of_strings('LEET', 'CODE') == ''
```
这三条正是 LC1071 的官方示例：整除重复型（'ABCABC' 是 'ABC' 的两倍）取基本块；'ABABAB'/'ABAB' 取长度 gcd(6,4)=2 的前缀 'AB'；完全不同源的 'LEET'/'CODE' 因拼接不可交换返回空串。三条合起来覆盖"整除、不整除但同源、不同源"三种关系。

```python
for a in range(-40, 41):
    for b in range(-40, 41):
        g, x, y = extended_gcd(a, b)
        assert g == gcd_euclid(a, b) == math.gcd(a, b)
        assert a * x + b * y == g
        assert lcm(a, b) == math.lcm(a, b)
```
这段是 −40..40 的全量穷举（81×81 = 6561 对，覆盖零、负、一正一负、双负所有组合）：gcd 与标准库 math.gcd 对齐；返回的 (x,y) 真的满足 a·x+b·y=g（这条最狠，直接验证符号修正的正确性）；lcm 与 math.lcm 对齐。这一片测试就是"符号与零"知识点的验收现场。

In [6]:
import math

assert gcd_euclid(54, 24) == 6 and gcd_euclid(0, 0) == 0

assert gcd_of_strings('ABCABC', 'ABC') == 'ABC'

assert gcd_of_strings('ABABAB', 'ABAB') == 'AB' and gcd_of_strings('LEET', 'CODE') == ''

for a in range(-40, 41):
    for b in range(-40, 41):
        g, x, y = extended_gcd(a, b)
        assert g == gcd_euclid(a, b) == math.gcd(a, b)
        assert a * x + b * y == g
        assert lcm(a, b) == math.lcm(a, b)

print('N126: 所有本章断言通过')

N126: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 求ax+by=gcd(a,b)一组解。

**练习 2：** 解释字符串gcd还需要拼接可交换条件。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（求 ax+by=gcd(a,b) 的一组解）**：思路就是把第三节 Step 3 的表格法走一遍——先写辗转相除的三列表（a、b、商、余数），再从最底下那行往回代，把每个余数都还原成 a、b 的线性组合，回代到头就得到一组 (x,y)。拿 (252,105) 手算应得 (−2,5)。边界要覆盖：a 或 b 为 0（比如 (7,0) 应返回 g=7, x=1, y=0）、两数为负（验证符号翻转）、(0,0)（返回 0,0,0 且等式成立）。最后与本章 extended_gcd 对拍即可；也可以再想想"通解怎么写"：在 (x₀,y₀) 的基础上，x 加 b/g、y 减 a/g 的任意整数倍仍然是一组解，这就是解不唯一的来源。

**练习 2（解释字符串 gcd 为什么还需要拼接可交换条件）**：论证方向是构造反例——只有"长度 gcd 切前缀"这一步的话，a='ABACADA'、b='A' 会输出 'A'，但 'ABACADA' 明显不是 'A' 重复出来的，答案错了；而加上 a+b==b+a 的检查后，'ABACADAA' 与 'AABACADA' 不等，函数正确返回空串。你要把"ab=ba 推出同源基本块"这层道理讲明白（提示：从两串的第一个不同位置推出周期性），再写清输入输出边界：任一串为空返回空串，长度互质但同源的例子（如 'ABAB' 与 'AB'，gcd=2）也要手算演示。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def gcd_euclid(a, b):
    a, b = (abs(a), abs(b))
    while b:
        a, b = (b, a % b)
    return a

def extended_gcd(a, b):
    old_r, r = (abs(a), abs(b))
    old_x, x = (1, 0)
    old_y, y = (0, 1)
    while r:
        q = old_r // r
        old_r, r = (r, old_r - q * r)
        old_x, x = (x, old_x - q * x)
        old_y, y = (y, old_y - q * y)
    return (old_r, old_x * (1 if a >= 0 else -1), old_y * (1 if b >= 0 else -1))

def lcm(a, b):
    return abs(a // gcd_euclid(a, b) * b) if a and b else 0

def gcd_of_strings(a, b):
    if not a or not b or a + b != b + a:
        return ''
    return a[:gcd_euclid(len(a), len(b))]

# 示例与回归测试
import math

assert gcd_euclid(54, 24) == 6 and gcd_euclid(0, 0) == 0

assert gcd_of_strings('ABCABC', 'ABC') == 'ABC'

assert gcd_of_strings('ABABAB', 'ABAB') == 'AB' and gcd_of_strings('LEET', 'CODE') == ''

for a in range(-40, 41):
    for b in range(-40, 41):
        g, x, y = extended_gcd(a, b)
        assert g == gcd_euclid(a, b) == math.gcd(a, b)
        assert a * x + b * y == g
        assert lcm(a, b) == math.lcm(a, b)

print('N126: 所有本章断言通过')

N126: 所有本章断言通过


## 11. 代表题与迁移

- **1979. Find Greatest Common Divisor of Array**：求数组中最小值与最大值的 gcd。这题练的是 gcd_euclid 本体，唯一的小弯是"先找 min/max 再求 gcd"。
- **1071. Greatest Common Divisor of Strings**：求两字符串的最大公因子串，gcd_of_strings 即题解本体。这题练的是"拼接可交换判据 + 长度 gcd 切前缀"这套组合拳。
- **1492. The kth Factor of n**：按从小到大输出 n 的第 k 个约数。这题练的是"约数"这个知识点：只用试除 1..√n，每发现一个小因子 d 就同时收获 d 和 n/d 这一对，按序收集到第 k 个即可。

三题由浅入深：1979 是 gcd 的直接调用，1071 把 gcd 的对象从数换成了字符串，1492 则从"最大公"转向"约数枚举"，合起来正好覆盖本章知识点的三个方向。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。